In [ ]:
!pip install google-generativeai gspread pandas httpx gspread-dataframe oauth2client google-api-python-client

In [ ]:
import os
import pandas as pd
import re
import numpy as np
import logging
import json
import pytz
from google.colab import drive
import gspread_dataframe as gd
import gspread
from gspread_dataframe import set_with_dataframe
from google.auth import default
from google.colab import auth
auth.authenticate_user()
from google.auth.transport.requests import Request
import ast
from google.auth import default
from google import genai
from google.genai import types
import httpx
import mimetypes
from googleapiclient.discovery import build
from io import BytesIO
from google.colab import auth
from googleapiclient.http import MediaIoBaseDownload
from io import StringIO
import time
import datetime


In [ ]:
# prompt: Search at the column D on this spreadsheet: https://docs.google.com/spreadsheets/d/1S_Brk-BpJlX9LoPLpbHERVAJuhx3Qi1RSNfdGovy7T0/edit?usp=sharing at the "gemini" table for URLs.
# I want to call the Gemini API to look into each URL where each one of them could be possible a PDF or a JPEG or other format of IMG. Those documents are essentially documents with CID of different diseases.
# I want the Gemini to look in the document and to find the CID and fill the Column G of this spreadsheet with the correspondent CID of the line of the document.
# My prompt is:
# Find the ICD in the attached file and provide only that. If you cannot find it, return 'N/A'. The document is in portuguese, so ICD is equal to CID in portuguese.

auth.authenticate_user()
drive_service = build('drive', 'v3')  # Corrigindo a criação do serviço

# Authenticate with Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the Google Sheet and select the worksheet
spreadsheet_url = '' - ### csv link ou filepath
sh = gc.open_by_url(spreadsheet_url)
worksheet = sh.worksheet("gemini2") # Replace with your actual sheet name

# Get all values from the worksheet
data = worksheet.get_all_values()

# Convert to a pandas DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# ================ LEITURA DA PLANILHA CID_DATA ================
print("\n📂 Lendo lista de CIDs PCD...")

# Acessar a aba 'cid_data'
worksheet_cid = sh.worksheet("cid_data")
cid_pcd_df = pd.DataFrame(worksheet_cid.get_all_records())

# Criar lista de CIDs válidos
cids_pcd = cid_pcd_df['CID'].str.strip().str.upper().tolist()
print(f"✅ {len(cids_pcd)} CIDs PCD carregados | Exemplo: {cids_pcd[:3]}")

# Initialize Gemini API client. Replace with your actual project ID
os.environ["GOOGLE_API_KEY"] = "####" #

client = genai.Client()


📂 Lendo lista de CIDs PCD...
✅ 152 CIDs PCD carregados | Exemplo: ['B20.0', 'B20.1', 'B20.2']


In [ ]:
# #################################################################################
# # =================== TRATAMENTO DO cid_candidato COM GEMINI ===================#
# #################################################################################

def tratar_cid3_com_gemini(texto):
    """Processa cid_candidato usando Gemini para extração refinada"""
    try:
        # Verificar valores vazios
        if pd.isna(texto) or str(texto).strip() in ('', 'N/A', 'nan'):
            print("DEBUG: Valor vazio - Retornando N/A")
            return 'N/A'

        print(f"\n🔍 DEBUG - Iniciando processamento do CID3:")
        print(f"Texto original: {texto}")

        # Construir prompt específico
        prompt = f"""Analise este texto e extraia APENAS O PRIMEIRO CÓDIGO CID VÁLIDO seguindo estas regras:
        1. Formato deve ser Letra + 2-3 números + opcional .1 decimal (Ex: A00, B34.1, F84.0)
        2. Ignore qualquer outro texto, números ou códigos diferentes
        3. Se não encontrar nenhum CID válido, responda com 'N/A'
        4. Confira com a lista internacional de CID.
        5. As vezes pode vir algo como H78-4, mas o hífen é um ponto.
        6. São muitos formatos diferentes, então é necessário sempre checar a similaridade com o que estiver mais próximo da lista de CID.

        Texto para análise: {texto}"""

        # Chamar Gemini
        print("🔄 Enviando para Gemini...")
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=[prompt]
        )

        # Debug da resposta bruta
        print(f"🔴 Resposta bruta do Gemini: {response.text}")

        # Processar resposta
        cid_encontrado = re.search(r'\b([A-Z]\d{2,3}(?:\.\d)?)\b', response.text)
        resultado = cid_encontrado.group(0) if cid_encontrado else 'N/A'

        print(f"✅ CID extraído: {resultado}")
        time.sleep(7)  # Delay entre requisições

        return resultado

    except Exception as e:
        print(f"❌ Erro crítico no processamento: {str(e)}")
        return 'N/A'

# Executar o processamento na coluna cid_candidato
print("\n" + "="*60)
print("INICIANDO PROCESSAMENTO DO cid_candidato COM GEMINI")
print("="*60)

df['cid_candidato_tratado'] = df['cid_candidato'].apply(tratar_cid3_com_gemini)
# Processar cid_candidato para cid_candidato_tratado
df['cid_candidato_tratado'] = df['cid_candidato'].apply(tratar_cid3_com_gemini)  # Passo existente

# Mostrar resultados
print("\n🔍 DEBUG - Amostra de resultados:")
print(df[['cid_candidato', 'cid_candidato_tratado']].head(5).to_markdown(index=False))


INICIANDO PROCESSAMENTO DO cid_candidato COM GEMINI

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: F84
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84

✅ CID extraído: F84

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: F84
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84

✅ CID extraído: F84

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: CID-10 F84
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84

✅ CID extraído: F84

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: F84 TEA
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84

✅ CID extraído: F84

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: f84.0
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84.0

✅ CID extraído: F84.0

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: F84
🔄 Enviando para Gemini...
🔴 Resposta bruta do Gemini: F84

✅ CID extraído: F84

🔍 DEBUG - Iniciando processamento do CID3:
Texto original: F84
🔄 Envian

In [ ]:
print(client)

In [ ]:
# Configurações iniciais
file_id = '### ' ####### INSERT THE cidlist2.csv

# 1. Download do arquivo
print("🔽 Fazendo download do arquivo...")
request = drive_service.files().get_media(fileId=file_id)
fh = BytesIO()
downloader = MediaIoBaseDownload(fh, request)

done = False
while not done:
    status, done = downloader.next_chunk()
    print(f"Progresso: {int(status.progress() * 100)}%")

# 2. Leitura do CSV com tratamento especial
fh.seek(0)
print("\n📖 Lendo arquivo CSV...")

# Detectar encoding e delimitador
try:
    # Ler primeiras linhas para análise
    sample = fh.read(1024).decode('utf-8-sig')
    fh.seek(0)

    delimiter = ';' if ';' in sample else ','
    print(f"Detectado delimitador: {delimiter}")

    # Ler CSV corrigindo cabeçalho
    cid_ref_df = pd.read_csv(
        fh,
        sep=delimiter,
        skiprows=1,
        header=None,
        names=['cidgroup', 'cidcode', 'descricao'],
        dtype=str,
        on_bad_lines='warn',
        encoding='utf-8-sig'
    ).fillna('')

except UnicodeDecodeError:
    fh.seek(0)
    cid_ref_df = pd.read_csv(
        fh,
        sep=delimiter,
        skiprows=1,
        header=None,
        names=['cidgroup', 'cidcode', 'descricao'],
        dtype=str,
        encoding='latin-1',
        on_bad_lines='warn'
    ).fillna('')

# 3. Função de processamento aprimorada
def processar_codigo(codigo):
    """Processa um código CID retirando espaços e normalizando"""
    codigo = str(codigo).strip().upper()

    # Remover caracteres especiais exceto letras, números e pontos
    return re.sub(r'[^A-Z0-9.]', '', codigo)

# 4. Construção do dicionário com verificação rigorosa
cid_dict = {}
for idx, row in cid_ref_df.iterrows():
    try:
        descricao = row['descricao'].strip()

        # Processar cidgroup
        if row['cidgroup']:
            grupo = processar_codigo(row['cidgroup'])
            if '-' in grupo:
                inicio, fim = grupo.split('-')
                letra = inicio[0]
                for num in range(int(inicio[1:]), int(fim[1:])+1):
                    codigo = f"{letra}{num:02d}"
                    cid_dict[codigo] = descricao
            else:
                codigo = processar_codigo(grupo)
                if codigo:
                    cid_dict[codigo] = descricao

        # Processar cidcode
        if row['cidcode']:
            for codigo in str(row['cidcode']).split(','):
                codigo_limpo = processar_codigo(codigo)
                if codigo_limpo and codigo_limpo not in cid_dict:
                    cid_dict[codigo_limpo] = descricao

    except Exception as e:
        print(f"⚠️ Erro linha {idx+1}: {str(e)}")
        continue

# 5. Verificação final
print("\n✅ Validação Final:")
print(f"Total de códigos mapeados: {len(cid_dict)}")
print("Exemplo F84.1:", cid_dict.get('F84.1', 'Não encontrado'))
print("Exemplo A00:", cid_dict.get('A00', 'Não encontrado'))
print("Exemplo G40.901:", cid_dict.get('G40.901', 'Não encontrado'))

# Mostrar amostra real dos dados carregados
print("\n🔍 Amostra do DataFrame carregado:")
print(cid_ref_df.head(5).to_markdown(index=False))


# *** Verificar/Criar colunas necessárias ***
colunas_necessarias = ['cid_extract-1', 'cid_extract-2', 'cid_extract-1-descricao', 'cid_extract-2-descricao']
for col in colunas_necessarias:
    if col not in df.columns:
        df[col] = ''




##############################################################################
# =================== PROCESSAR AS URLS / MANDAR PARA O GEMINI ============= #
##############################################################################


# ================ BLOCO COMPLETO ATUALIZADO ================
def processar_url(url):
    # Inicializa resultado com valores padrão
    resultado = {
        'cid_extract-1': 'N-A',
        'cid_extract-2': '',
        'cid_extract-1-descricao': 'N-A',
        'cid_extract-2-descricao': '',
        'Data': 'N/A',
        'CRM': 'N/A',
        'cid_texto': 'N/A',
        'tipo_laudo': 'indeterminado'
    }

    try:
        print(f"\n⏳ Processando URL: {url}")

        # Etapa 1: Download do arquivo
        print("⌛ Baixando arquivo...")
        response = httpx.get(url, timeout=30)
        response.raise_for_status()

        # Etapa 2: Processamento do conteúdo
        file_part = types.Part.from_bytes(
            data=response.content,
            mime_type='application/pdf' if url.lower().endswith('.pdf') else f'image/{url.split(".")[-1].lower()}'
        )

        # Etapa 3: Chamada ao Gemini
        prompt = """Extraia as seguintes informações do documento:

    -   CIDs (Exemplo: F71, F84.1, G40.901, CID10-F84.4, CID10: F42.4 ...)
    -   Data mais relevante (formato: dd/mm/aaaa)
    -   Número do CRM e região
    -   Indicação se o laudo é feito à mão ou digitado

    Retorne UM ÚNICO JSON (sem markdown) com as chaves:
    {
        "cids": [
            {
                "cid": "CID_ENCONTRADA",
                "texto_referencia": "TEXTO_ONDE_A_CID_FOI_ENCONTRADA"
            },
            ...
        ],
        "data": "data_encontrada",
        "crm": "numero_crm",
        "tipo_laudo": "mão" ou "digitado" ou "indeterminado"
    }
    Se algum campo não for encontrado, use null.
    Se não for possível determinar o tipo de laudo, use "indeterminado". """
        gemini_response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=[file_part, prompt]
        )

        # Etapa 4: Processamento da resposta
        try:
            resposta_limpa = re.sub(r'```json|```|\*+', '', gemini_response.text)
            dados = json.loads(resposta_limpa)

            # Extração de CIDs
            cids = []
            textos = []
            for item in dados.get('cids', []):
                cid = re.sub(r'[^A-Z0-9.]', '', str(item.get('cid', '')).upper())
                if cid:
                    cids.append(cid)
                    textos.append(item.get('texto_referencia', 'Sem contexto'))

            # Atualizar resultado
            resultado.update({
                'cid_texto': ' | '.join(textos),
                'tipo_laudo': dados.get('tipo_laudo', 'indeterminado'),
                'Data': dados.get('data', 'N/A'),
                'CRM': str(dados.get('crm', 'N/A')).strip()
            })

            # Mapear CIDs válidos
            if cids:
                resultado['cid_extract-1'] = cids[0]
                resultado['cid_extract-1-descricao'] = cid_dict.get(cids[0], 'N-A')

                if len(cids) > 1:
                    resultado['cid_extract-2'] = ', '.join(cids[1:])
                    resultado['cid_extract-2-descricao'] = ', '.join([cid_dict.get(c, 'N-A') for c in cids[1:]])

        except Exception as parse_error:
            print(f"Erro no parse: {str(parse_error)}")
            # Fallback com regex
            cids = re.findall(r'\b[A-Z]\d{2,3}(?:\.\d+)?\b', gemini_response.text)
            if cids:
                resultado['cid_extract-1'] = cids[0]
                resultado['cid_extract-1-descricao'] = cid_dict.get(cids[0], 'N-A')

        return resultado

    except Exception as e:
        print(f"❌ [{datetime.datetime.now().strftime('%H:%M:%S')}] Erro geral: {str(e)}")
        return resultado

# ================ ATUALIZAR DATAFRAME ================

# Atualizar todas as colunas
        colunas_para_atualizar = [
            'cid_extract-1', 'cid_extract-2',
            'cid_extract-1-descricao', 'cid_extract-2-descricao',
            'Data', 'CRM', 'cid_texto', 'tipo_laudo'  # Garantir todas
        ]

# Adicionar novas colunas se não existirem
for col in ['Data', 'CRM']:
    if col not in df.columns:
        df[col] = ''

# Processar todas as URLs
print("\n" + "="*50)
print("INICIANDO PROCESSAMENTO DE DOCUMENTOS")
print("="*50)

for index, row in df.iterrows():
    try:
        url = row['Laudo Médico']
        if pd.notna(url) and url.strip().lower() not in ("", "url"):
            print(f"\n📌 Linha {index+2}:")

            # Reinicializar resultado a cada iteração
            resultado = {
                'cid_extract-1': 'N-A',
                'cid_extract-2': '',
                'cid_extract-1-descricao': 'N-A',
                'cid_extract-2-descricao': '',
                'Data': 'N/A',
                'CRM': 'N/A',
                'cid_texto': 'N/A',
                'tipo_laudo': 'indeterminado'
            }

            resultado = processar_url(url)

            # Atualizar todas as colunas
            for col in ['cid_extract-1', 'cid_extract-2',
                       'cid_extract-1-descricao', 'cid_extract-2-descricao',
                       'Data', 'CRM', 'cid_texto', 'tipo_laudo']:
                df.at[index, col] = resultado[col]

            print(f"✅ Dados atualizados: {resultado}")
            time.sleep(5)

        else:
            print(f"\n⏭ Linha {index+2} pulada - URL inválida")

    except Exception as e:
        print(f"🔥 Erro crítico na linha {index+2}: {str(e)}")
        continue

####################### ===================== ########################
########### VALIDAR INPUT DO USUÁRIO            ######################
########### ======================================= ##################


def validar_cid(row):
    """Verifica se o cid_candidato_tratado coincide com CID1 ou CID2"""
    cid3 = str(row['cid_candidato_tratado']).strip()

    # Caso base para inválido
    if cid3 in ('N/A', '', 'nan'):
        return 'FALSE'

    # Verificar contra CID1
    if cid3 == str(row['cid_extract-1']).strip():
        return 'TRUE'

    # Verificar contra CID2 (considerando múltiplos CIDs separados por vírgula)
    cid2_list = [cid.strip() for cid in str(row['cid_extract-2']).split(',') if cid.strip()]
    return 'TRUE' if cid3 in cid2_list else 'FALSE'

  # ================ VALIDAÇÃO CRUZADA ================
print("\n🔍 Validando cid_candidato_tratado contra CID1/CID2...")

# Criar/sobrescrever coluna de validação
df['validacao_cruzada'] = df.apply(validar_cid, axis=1)


# [...] Após o processamento das URLs:

# Nova validação cruzada (NOVO!)
df['validacao_cruzada'] = df.apply(validar_cid, axis=1)  # ← Adicione aqui

def verificar_enquadramento(row):
    """Verifica se o cid_candidato_tratado está na lista PCD"""
    if row['validacao_cruzada'] != 'TRUE':
        return 'N/A'  # Só valida se a validação anterior for TRUE

    cid3 = str(row['cid_candidato_tratado']).strip().upper()
    return 'TRUE' if cid3 in cids_pcd else 'FALSE'

# ================ VERIFICAÇÃO DE ENQUADRAMENTO ================
print("\n🔍 Verificando enquadramento na lei PCD...")
df['enquadramento'] = df.apply(verificar_enquadramento, axis=1)

# Pré-visualização atualizada
print("\n" + "="*50)
print("PRÉ-VISUALIZAÇÃO DOS DADOS:")
print(df[['cid_extract-1', 'cid_extract-2', 'cid_candidato_tratado', 'validacao_cruzada']].head(10).to_markdown(index=False))

🔽 Fazendo download do arquivo...
Progresso: 100%

📖 Lendo arquivo CSV...
Detectado delimitador: ;

✅ Validação Final:
Total de códigos mapeados: 27282
Exemplo F84.1: Autismo atípico
Exemplo A00: Doenças infecciosas intestinais
Exemplo G40.901: Não encontrado

🔍 Amostra do DataFrame carregado:
| cidgroup   | cidcode   | descricao                                                                                      |
|:-----------|:----------|:-----------------------------------------------------------------------------------------------|
| A00        | B99       | Capítulo I - Algumas doenças infecciosas e parasitárias                                        |
| C00        | D48       | Capítulo II - Neoplasias [tumores]                                                             |
| D50        | D89       | Capítulo III  - Doenças do sangue e dos órgãos hematopoéticos e alguns transtornos imunitários |
| E00        | E90       | Capítulo IV - Doenças endócrinas, nutricionais e metabólica

In [ ]:
##########################################################################################
# # =================== ADICIONAR DESCRIÇÃO PARA cid_candidato_tratado ===================
##########################################################################################

print("\n" + "="*60)
print("PROCESSANDO DESCRIÇÕES PARA cid_candidato_tratado")
print("="*60)

# Criar nova coluna se não existir
if 'cid_candidato_tratado_descr' not in df.columns:
    df['cid_candidato_tratado_descr'] = ''

# Função de busca com debug
def buscar_descricao_cid3(cid):
    try:
        cid_limpo = cid.strip().upper()

        # Debug inicial
        print(f"\n🔍 Buscando CID3: '{cid}' -> Normalizado: '{cid_limpo}'")

        if cid_limpo in ('N/A', '', 'NAN'):
            print(f"❌ CID3 vazio/inválido")
            return 'N/A'

        # Buscar no dicionário
        descricao = cid_dict.get(cid_limpo, 'N/A')

        # Debug do resultado
        if descricao != 'N/A':
            print(f"✅ Encontrado: {descricao}")
        else:
            print(f"⚠️ CID não encontrado no dicionário: {cid_limpo}")

        return descricao

    except Exception as e:
        print(f"Erro ao buscar CID3 {cid}: {str(e)}")
        return 'N/A'

# Aplicar função em toda a coluna
df['cid_candidato_tratado_descr'] = df['cid_candidato_tratado'].apply(buscar_descricao_cid3)

# Estatísticas de processamento
total_cid3 = len(df['cid_candidato_tratado'])
encontrados = sum(df['cid_candidato_tratado_descr'] != 'N/A')
print("\n" + "="*60)
print(f"RESUMO cid_candidato_tratado_descr:")
print(f"Total de CIDs processados: {total_cid3}")
print(f"CIDs encontrados no dicionário: {encontrados} ({encontrados/total_cid3:.1%})")
print(f"CIDs não encontrados: {total_cid3 - encontrados}")
print("="*60)

# Amostra de resultados
print("\n🔍 DEBUG - Amostra final:")
print(df[['cid_candidato_tratado', 'cid_candidato_tratado_descr']].head(5).to_markdown(index=False))


PROCESSANDO DESCRIÇÕES PARA cid_candidato_tratado

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84.0' -> Normalizado: 'F84.0'
✅ Encontrado: Autismo infantil

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'F84' -> Normalizado: 'F84'
✅ Encontrado: Transtornos globais do desenvolvimento

🔍 Buscando CID3: 'N/A' -> Normalizado: 'N/A'
❌ CID3 vazio/inválido

RESUMO cid_candidato_tratado_descr:
Total de CIDs processados: 9
CIDs encontrados no dicionário: 8 (88.9%)

In [ ]:
# Atualizar planilha automaticamente
print("\n⌛ Atualizando planilha...")
worksheet.clear()
set_with_dataframe(worksheet, df)
print("✅ Planilha atualizada com sucesso!")


⌛ Atualizando planilha...
✅ Planilha atualizada com sucesso!
